In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import roc_curve, auc, roc_auc_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import math

In [ ]:
class LensingPretrainDataset(Dataset):
    """Dataset for MAE pre-training - only no_sub class, no labels needed"""
    def __init__(self, data_dir):
        self.samples = []
        no_sub_path = os.path.join(data_dir, 'no_sub')
        for fname in os.listdir(no_sub_path):
            if fname.endswith('.npy'):
                self.samples.append(os.path.join(no_sub_path, fname))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = np.load(self.samples[idx], allow_pickle=True)
        if img.dtype == object:
            img = img[0]
        img = img.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = torch.tensor(img).unsqueeze(0)
        return img


class LensingClassifyDataset(Dataset):
    """Dataset for fine-tuning - all 3 classes with labels"""
    def __init__(self, data_dir):
        self.samples = []
        self.labels = []
        self.class_map = {'no_sub': 0, 'cdm': 1, 'axion': 2}

        for class_name, label in self.class_map.items():
            class_path = os.path.join(data_dir, class_name)
            for fname in os.listdir(class_path):
                if fname.endswith('.npy'):
                    self.samples.append(os.path.join(class_path, fname))
                    self.labels.append(label)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = np.load(self.samples[idx], allow_pickle=True)
        if img.dtype == object:
            img = img[0]
        img = img.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = torch.tensor(img).unsqueeze(0)
        return img, self.labels[idx]


# UPDATE THIS PATH
DATA_PATH

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=64, patch_size=8, in_channels=1, embed_dim=256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return

In [ ]:
optimizer = torch.optim.AdamW(mae_model.parameters(), lr=1.5e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

num_epochs = 50

for epoch in range(num_epochs):
    mae_model.train()
    running_loss = 0.0

    pbar = tqdm(pretrain_loader, desc=f"Pre-train Epoch {epoch+1}/{num_epochs}")
    for images in pbar:
        images = images.to(device)

        optimizer.zero_grad()
        loss, _, _ = mae_model(images)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postf

In [ ]:
mae_model.eval()
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

with torch.no_grad():
    sample_imgs = next(iter(pretrain_loader))[:3].to(device)

    for i in range(3):
        img = sample_imgs[i:i+1]
        loss, pred, mask = mae_model(img)

        axes[i, 0].imshow(img[0, 0].cpu().numpy(), cmap='gray')
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')

        mask_img = img[0, 0].cpu().clone()
        mask_2d = mask[0].reshape(8, 8).repeat_interleave(8, 0).repeat_interleave(8, 1)
        mask_img[mask_2d.bool()] = 0.5
        axes[i, 1].imshow(mask_img.numpy(), cmap='gray', vmin=0, vmax=1)
        axes[i, 1].set_title('Masked (75%)')
        axes[i, 1].axis('off')

        pred_img = pred[0].reshape(8, 8, 8, 8).permute(0, 2, 1, 3).reshape(64, 64)
        composite = img[0, 0].cpu().clone()
        composite[mask_2d.bool()] = pred_img.cpu()[mask_2d.bool()]
        composite = composite.clamp(0, 1)
        axes[i, 2].imshow(composite.numpy(), cmap='gray', vmin=0, vmax=1)
        axes[i, 2].set_title('Reconstructed')
        axes[i, 2].axis('off')

plt.suptitle('MAE Reconstruction: Original -> Masked -> Reconstructed', fontsize=14)
plt.tight_layout()
plt.savefig('mae_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
class MAEClassifier(nn.Module):
    def __init__(self, mae_model, num_classes=3):
        super().__init__()
        self.patch_embed = mae_model.patch_embed
        self.cls_token = mae_model.cls_token
        self.pos_embed = mae_model.pos_embed
        self.encoder_blocks = mae_model.encoder_blocks
        self.encoder_norm = mae_model.encoder_norm

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :]

        cls = self.cls_token + self.pos_embed[:, :1, :]
        cls = cls.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)

        for block in self.encoder_blocks:
            x = block(x)
        x = self.encoder_norm(x)

        cls_output = x[:, 0]
        return self.classifier(cls_output)

classifier = MAEClassifier(mae_model).to(device)
print(f"Classifier parameters: {sum(p.numel() for p in classifier.parameters()):,}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

num_epochs = 25

for epoch in range(num_epochs):
    classifier.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Fine-tune Epoch {epoch+1}/{num_epochs}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = classifier(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.1f}%'
        })

    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {running_loss/len(train_loader):.4f} "
          f"- Acc: {100.*correct/total:.2f}%\n")

In [ ]:
classifier.eval()
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(device)
        outputs = classifier(images)
        probs = torch.softmax(outputs, dim=1)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

class_names = ['No Substructure', 'CDM', 'Axion']
plt.figure(figsize=(10, 8))

for i, name in enumerate(class_names):
    binary_labels = (all_labels == i).astype(int)
    fpr, tpr, _ = roc_curve(binary_labels, all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')
    print(f"{name}: AUC = {roc_auc:.4f}")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Foundation Model Classification (Test IX.A)')
plt.legend()
plt.grid(True)
plt.savefig('roc_curves_test9a.png', dpi=150, bbox_inches='tight')
plt.show()

macro_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
print(f"\nMacro AUC: {macro_auc:.4f}")

In [ ]:
torch.save(classifier.state_dict(), 'mae_classifier.pth')
print("Classifier saved!")